In [18]:

import re 
from rdkit import Chem
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem import Draw
IPythonConsole.ipython_useSVG=True 
IPythonConsole.drawOptions.addAtomIndices = True

import pyarrow.dataset as ds

from smiles_blocks.retrosynthesis import  retrosynthetic_analysis,prep_rbrics_data
from smiles_blocks.smiles_fragmentation import (process_set_smiles,
                                                get_semantic_mem_score,
                                                SmilesRegex)
from smiles_blocks.files import PROCESSED_DATA_DIR

In [2]:
# source : https://pubchem.ncbi.nlm.nih.gov/compound/Celecoxib#section=SMILES
celecoxib_smiles = "CC1=CC=C(C=C1)C2=CC(=NN2C3=CC=C(C=C3)S(=O)(=O)N)C(F)(F)F"
celecoxib_mol = Chem.MolFromSmiles(celecoxib_smiles)

In [3]:
smiles_regex= SmilesRegex()

In [4]:
chemical_dict= prep_rbrics_data()

In [5]:
retrosynth_results = retrosynthetic_analysis(celecoxib_smiles, chemical_dict)

In [6]:
retrosynth_results

defaultdict(<function smiles_blocks.retrosynthesis.annotate_rbond.<locals>.<lambda>()>,
            {frozenset({21, 24}): defaultdict(set, {24: {'L8'}, 21: {'L14'}}),
             frozenset({19, 23}): defaultdict(set, {23: {'L9'}, 19: {'L16'}}),
             frozenset({20,
                        25}): defaultdict(set, {25: {'L12b'}, 20: {'L16'}}),
             frozenset({18,
                        22}): defaultdict(set, {22: {'L14'}, 18: {'L16'}})})

In [7]:
rd_smiles= set(
    Chem.MolToRandomSmilesVect(celecoxib_mol,
                                      200_000,
                                      isomericSmiles=False,
                                      kekuleSmiles=True,
                                      allBondsExplicit=True,)
)

In [8]:
mem_rd_smiles= [
    (smiles,get_semantic_mem_score(smiles, smiles_regex.regex)) for smiles in rd_smiles
]

In [9]:
mem_rd_smiles

[('C1(=C-C=C(-C=C-1)-C1=C-C(=N-N-1-C1-C=C-C(=C-C=1)-S(=O)(=O)-N)-C(-F)(-F)-F)-C',
  np.float64(2.25)),
 ('C1(=C-C=C(-C2-N(-N=C(-C=2)-C(-F)(-F)-F)-C2=C-C=C(-C=C-2)-S(-N)(=O)=O)-C=C-1)-C',
  np.float64(3.551282051282051)),
 ('C1=C(-C2-C=C-C(=C-C=2)-C)-N(-N=C-1-C(-F)(-F)-F)-C1-C=C-C(=C-C=1)-S(=O)(=O)-N',
  np.float64(1.618421052631579)),
 ('C(-F)(-F)(-C1-C=C(-C2=C-C=C(-C=C-2)-C)-N(-N=1)-C1-C=C-C(=C-C=1)-S(=O)(-N)=O)-F',
  np.float64(2.2435897435897436)),
 ('C1(-C2-C=C-C(-C)=C-C=2)-N(-C2-C=C-C(-S(=O)(-N)=O)=C-C=2)-N=C(-C(-F)(-F)-F)-C=1',
  np.float64(2.6794871794871793)),
 ('C(-F)(-C1=N-N(-C(-C2-C=C-C(-C)=C-C=2)=C-1)-C1-C=C-C(-S(-N)(=O)=O)=C-C=1)(-F)-F',
  np.float64(2.769230769230769)),
 ('O=S(=O)(-C1=C-C=C(-C=C-1)-N1-C(-C2-C=C-C(=C-C=2)-C)=C-C(=N-1)-C(-F)(-F)-F)-N',
  np.float64(2.263157894736842)),
 ('C1=C(-C=C-C(=C-1)-N1-N=C(-C(-F)(-F)-F)-C=C-1-C1-C=C-C(-C)=C-C=1)-S(=O)(-N)=O',
  np.float64(1.9210526315789473)),
 ('S(=O)(-N)(=O)-C1=C-C=C(-C=C-1)-N1-C(-C2-C=C-C(-C)=C-C=2)=C-C(=N-1)-C(-F

In [10]:
sorted_rd_smiles = sorted(mem_rd_smiles, key=lambda x: x[1])

In [11]:
len(sorted_rd_smiles)

13424

In [12]:
success,blocked_smiles,blocks= process_set_smiles(sorted_rd_smiles,retrosynth_results)

In [13]:
blocked_smiles

{'smiles': 'C1(-C=C-C(-C2-N(-C3=C-C=C(-S(-N)(=O)=O)-C=C-3)-N=C(-C(-F)(-F)-F)-C=2)=C-C=1)-C',
 'smiles_blocked': 'F-C(-F)(-F).C1=N-N(-C(-C2=C-C=C(-C=C-2)-C)=C-1).C1-C=C-C(=C-C=1).S(-N)(=O)=O',
 'mem_score': np.float64(1.736842105263158),
 'unique_id_seq': '0_FC(F)F_1.7_Cc1ccc(-c2ccn[nH]2)cc1_9.0_c1ccccc1_3.1_N[SH](=O)=O_3',
 'retro_bond_ratio': 0.75,
 'nb_block_cq_ok': 3}

In [14]:
blocks

[{'block': 'F-C(-F)(-F)',
  'can_smiles': 'FC(F)F',
  'first_connected_can_idx': 0,
  'last_connected_can_idx': 1,
  'unique_id': '0_FC(F)F_1',
  'begin_tag': 'no_tag',
  'end_tag': 'L8',
  'MolWt': 70.003034692,
  'nHDonors': 0,
  'nHAcceptors': 0,
  'nRotatableBonds': 0,
  'CrippenlogP': 1.1784999999999999,
  'TPSA': 0.0,
  'status': True},
 {'block': 'C1=N-N(-C(-C2=C-C=C(-C=C-2)-C)=C-1)',
  'can_smiles': 'Cc1ccc(-c2ccn[nH]2)cc1',
  'first_connected_can_idx': 7,
  'last_connected_can_idx': 9,
  'unique_id': '7_Cc1ccc(-c2ccn[nH]2)cc1_9',
  'begin_tag': 'L14',
  'end_tag': 'L9',
  'MolWt': 158.08439832,
  'nHDonors': 1,
  'nHAcceptors': 1,
  'nRotatableBonds': 1,
  'CrippenlogP': 2.3851200000000006,
  'TPSA': 28.68,
  'status': True},
 {'block': 'C1-C=C-C(=C-C=1)',
  'can_smiles': 'c1ccccc1',
  'first_connected_can_idx': 0,
  'last_connected_can_idx': 3,
  'unique_id': '0_c1ccccc1_3',
  'begin_tag': 'L16',
  'end_tag': 'L16',
  'MolWt': 78.046950192,
  'nHDonors': 0,
  'nHAcceptors': 0